# Dasar Machine Learning

## Course 3 Bagian d: Decision Tree

Data kualitas anggur dipakai untuk dua tugas. Pertama, klasifikasi warna anggur memakai seluruh fitur kimia. Kedua, regresi kadar gula residu.

### Tujuan

- Melatih pohon keputusan tanpa batas kedalaman.
- Mengukur tingkat overfitting lewat perbandingan galat latih dan uji.
- Memperbaiki model dengan grid search dan validasi silang.
- Menampilkan struktur pohon hasil pelatihan.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Soal 1: Memuat data

Sebagian baris dipakai agar pelatihan tetap cepat.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/Wine_Quality_Data.csv"
)
SAMPLE_SIZE = 2000

wine_data = pd.read_csv(DATA_URL, sep=",").sample(
    n=SAMPLE_SIZE, random_state=42
).reset_index(drop=True)
wine_data.head()


In [ ]:
wine_data.dtypes


Kolom warna diubah menjadi angka: putih menjadi 0 dan merah menjadi 1.


In [ ]:
wine_data["color"] = wine_data["color"].map({"white": 0, "red": 1}).astype(int)
wine_data["color"].value_counts()


## Soal 2: Pembagian data

`StratifiedShuffleSplit` menjaga proporsi kelas pada data latih dan uji.


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

feature_columns = [name for name in wine_data.columns if name != "color"]
splitter = StratifiedShuffleSplit(n_splits=1, test_size=500, random_state=42)
train_index, test_index = next(
    splitter.split(wine_data[feature_columns], wine_data["color"])
)

X_train = wine_data.loc[train_index, feature_columns]
y_train = wine_data.loc[train_index, "color"]
X_test = wine_data.loc[test_index, feature_columns]
y_test = wine_data.loc[test_index, "color"]


In [ ]:
y_train.value_counts(normalize=True).sort_index()


In [ ]:
y_test.value_counts(normalize=True).sort_index()


## Soal 3: Pohon tanpa batas

Pohon dilatih tanpa batas kedalaman, jumlah fitur, maupun jumlah daun.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

full_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
full_tree.tree_.node_count, full_tree.tree_.max_depth


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def measure_error(actual, predicted, label):
    return pd.Series(
        {
            "accuracy": accuracy_score(actual, predicted),
            "precision": precision_score(actual, predicted),
            "recall": recall_score(actual, predicted),
            "f1": f1_score(actual, predicted),
        },
        name=label,
    )


In [ ]:
pd.concat(
    [
        measure_error(y_train, full_tree.predict(X_train), "train"),
        measure_error(y_test, full_tree.predict(X_test), "test"),
    ],
    axis=1,
)


Selisih kecil antara skor latih dan uji menandakan overfitting ringan. Pohon yang terlalu dalam cenderung menghafal data latih.


## Soal 4: Grid search dengan validasi silang

Kedalaman maksimum dan jumlah fitur yang dipertimbangkan diuji pada beberapa kombinasi. Kombinasi dengan akurasi terbaik dipilih otomatis.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": range(1, full_tree.tree_.max_depth + 1, 2),
    "max_features": range(1, 4),
}

search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
search.best_params_


In [ ]:
search.best_estimator_.tree_.node_count, search.best_estimator_.tree_.max_depth


In [ ]:
pd.concat(
    [
        measure_error(y_train, search.predict(X_train), "train"),
        measure_error(y_test, search.predict(X_test), "test"),
    ],
    axis=1,
)


## Soal 5: Regresi kadar gula residu

Target diganti menjadi `residual_sugar` yang bersifat kontinu, sehingga metrik yang dipakai adalah mean squared error.


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

regression_features = [name for name in wine_data.columns if name != "residual_sugar"]
X_train = wine_data.loc[train_index, regression_features]
y_train = wine_data.loc[train_index, "residual_sugar"]
X_test = wine_data.loc[test_index, regression_features]
y_test = wine_data.loc[test_index, "residual_sugar"]

baseline_tree = DecisionTreeRegressor().fit(X_train, y_train)
regression_grid = {
    "max_depth": range(1, baseline_tree.tree_.max_depth + 1, 2),
    "max_features": range(1, 4),
}
regression_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=regression_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
regression_search.best_estimator_.tree_.node_count


In [ ]:
pd.Series(
    {
        "train": mean_squared_error(y_train, regression_search.predict(X_train)),
        "test": mean_squared_error(y_test, regression_search.predict(X_test)),
    },
    name="MSE",
).to_frame().T


In [ ]:
sns.set_context("notebook")
sns.set_style("white")

comparison = pd.DataFrame(
    {"test": y_test.values, "predict": regression_search.predict(X_test)}
).sort_values("test")

figure, axis = plt.subplots(figsize=(6, 6))
axis.scatter(comparison["test"], comparison["predict"], s=8)
axis.set(xlabel="Test", ylabel="Predict", xlim=(0, 35), ylim=(0, 35),
         title="Actual vs predicted residual sugar")
plt.show()


## Soal 6: Menampilkan struktur pohon

Struktur pohon digambar langsung dengan `plot_tree` dari scikit-learn, tanpa memerlukan GraphViz maupun PyDotPlus.


In [ ]:
from sklearn.tree import plot_tree

figure, axis = plt.subplots(figsize=(20, 10))
plot_tree(search.best_estimator_, feature_names=feature_columns,
          filled=True, max_depth=3, fontsize=8, ax=axis)
plt.show()


## Ringkasan

Pohon tanpa batas mudah overfitting. Pembatasan kedalaman dan jumlah fitur lewat grid search menurunkan galat uji. Untuk target kontinu, pohon regresi memerlukan kedalaman jauh lebih besar karena nilai targetnya bervariasi halus.
